# Processign Silver layer for res_partner model

In [0]:
%sql

DESCRIBE bakery_pos_develop.bronze.res_partner

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW stg_res_partner AS
SELECT
    id AS customer_id,
    l10n_latam_identification_type_id AS identification_type,
    vat AS identification_number,
    phone,
    email,
    CAST(create_date AS TIMESTAMP) AS created_at,
    CAST(write_date AS TIMESTAMP) AS updated_at,
    processing_timestamp AS bronze_ingested_at
FROM bakery_pos_develop.bronze.res_partner
QUALIFY ROW_NUMBER() OVER(
    PARTITION BY id
    ORDER BY write_date DESC,
    processing_timestamp DESC
) = 1



In [0]:
%sql

CREATE TABLE IF NOT EXISTS bakery_pos_develop.silver.res_partner (
    customer_id BIGINT,
    identification_type STRING,
    identification_number STRING,
    phone STRING,
    email STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    bronze_ingested_at TIMESTAMP,
    silver_processed_at TIMESTAMP,
    is_updated BOOLEAN,
    is_deleted BOOLEAN
) USING DELTA;

In [0]:
%sql

MERGE INTO bakery_pos_develop.silver.res_partner AS target
USING (
  SELECT *, current_timestamp() AS silver_processed_at
  FROM stg_res_partner
) AS source
ON target.customer_id = source.customer_id
WHEN MATCHED AND source.updated_at > target.updated_at THEN
  UPDATE SET
    identification_type = source.identification_type,
    identification_number = source.identification_number,
    phone = source.phone,
    email = source.email,
    created_at = source.created_at,
    updated_at = source.updated_at,
    is_updated = True,
    is_deleted = False,
    bronze_ingested_at = source.bronze_ingested_at,
    silver_processed_at = source.silver_processed_at
WHEN NOT MATCHED THEN
  INSERT (
    customer_id,
    identification_type,
    identification_number,
    phone,
    email,
    created_at,
    updated_at,
    bronze_ingested_at,
    silver_processed_at,
    is_updated,
    is_deleted
  )
  VALUES (
    source.customer_id,
    source.identification_type,
    source.identification_number,
    source.phone,
    source.email,
    source.created_at,
    source.updated_at,
    source.bronze_ingested_at,
    source.silver_processed_at,
    False,
    False
  );

In [0]:
%sql
describe bakery_pos_develop.silver.res_partner